In [1]:
import sys
import os

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import polars as pl
import MetaTrader5 as mt5
from src.infra.mtBase import mtBase
from src.infra.PredictionParser import PredictionParser, PredictionData

## 1. Initialize Parser and Load Predictions

In [2]:
# Initialize the prediction parser
parser = PredictionParser(predictions_dir="../predictions")

# Find all prediction files
prediction_files = parser.find_prediction_files()
print(f"Found {len(prediction_files)} prediction files:")
for file in prediction_files:
    print(f"  - {file.name}")

# Load all predictions into a list
all_predictions = []
for file in prediction_files:
    try:
        predictions = parser.parse_json_file(file)
        all_predictions.extend(predictions)
        print(f"✓ Loaded {len(predictions)} predictions from {file.name}")
    except Exception as e:
        print(f"✗ Error loading {file.name}: {e}")

print(f"\nTotal predictions loaded: {len(all_predictions)}")

# Display summary of loaded predictions
if all_predictions:
    print("\nPrediction Summary:")
    for i, pred in enumerate(all_predictions, 1):
        print(f"{i}. {pred}")

Found 1 prediction files:
  - prediction_mt5_15oct2025_001.json
✓ Loaded 4 predictions from prediction_mt5_15oct2025_001.json

Total predictions loaded: 4

Prediction Summary:
1. PredictionData(symbol='COF', last_training_day='2025-10-30', last_close_price=218.17, n_trading_days=5, score=1.3, magic=3611410359460303377, sl_pct=0.07, tp_pct=0.9, source=prediction_mt5_15oct2025_001.json)
2. PredictionData(symbol='SRPT', last_training_day='2025-10-30', last_close_price=23.25, n_trading_days=5, score=1.2, magic=4142139608164363334, sl_pct=0.07, tp_pct=0.9, source=prediction_mt5_15oct2025_001.json)
3. PredictionData(symbol='MU', last_training_day='2025-10-30', last_close_price=224.01, n_trading_days=5, score=1.25, magic=7256825577762653019, sl_pct=0.07, tp_pct=0.9, source=prediction_mt5_15oct2025_001.json)
4. PredictionData(symbol='FMC', last_training_day='2025-10-30', last_close_price=15.53, n_trading_days=5, score=1.1, magic=6170249002540558280, sl_pct=0.07, tp_pct=0.9, source=prediction_m

In [3]:
mtb = mtBase(
    account="mt5demo_acc_usd",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)
mtb.mt5_init()

MetaTrader 5 connection established


In [5]:
info = mt5.symbol_info("CHFSEK")._asdict()
print(info)

{'custom': False, 'chart_mode': 0, 'select': False, 'visible': False, 'session_deals': 0, 'session_buy_orders': 0, 'session_sell_orders': 0, 'volume': 0, 'volumehigh': 0, 'volumelow': 0, 'time': 0, 'digits': 5, 'spread': 0, 'spread_float': True, 'ticks_bookdepth': 32, 'trade_calc_mode': 0, 'trade_mode': 4, 'start_time': 0, 'expiration_time': 0, 'trade_stops_level': 0, 'trade_freeze_level': 0, 'trade_exemode': 2, 'swap_mode': 1, 'swap_rollover3days': 3, 'margin_hedged_use_leg': False, 'expiration_mode': 15, 'filling_mode': 1, 'order_mode': 127, 'order_gtc_mode': 0, 'option_mode': 0, 'option_right': 0, 'bid': 0.0, 'bidhigh': 0.0, 'bidlow': 0.0, 'ask': 0.0, 'askhigh': 0.0, 'asklow': 0.0, 'last': 0.0, 'lasthigh': 0.0, 'lastlow': 0.0, 'volume_real': 0.0, 'volumehigh_real': 0.0, 'volumelow_real': 0.0, 'option_strike': 0.0, 'point': 1e-05, 'trade_tick_value': 0.10526481997084165, 'trade_tick_value_profit': 0.10526481997084165, 'trade_tick_value_loss': 0.10546565194647409, 'trade_tick_size': 1

In [6]:
import MetaTrader5 as mt5
tick_dict = mtb.get_symbol_price("MU")
res = mt5.symbol_select("MU", True)
info = mt5.symbol_info("MU")
print(info)

SymbolInfo(custom=False, chart_mode=1, select=True, visible=True, session_deals=0, session_buy_orders=0, session_sell_orders=0, volume=114, volumehigh=92998, volumelow=100, time=1761955175, digits=2, spread=2, spread_float=True, ticks_bookdepth=0, trade_calc_mode=32, trade_mode=4, start_time=0, expiration_time=0, trade_stops_level=5, trade_freeze_level=0, trade_exemode=3, swap_mode=0, swap_rollover3days=3, margin_hedged_use_leg=False, expiration_mode=15, filling_mode=1, order_mode=127, order_gtc_mode=0, option_mode=0, option_right=0, bid=223.78, bidhigh=231.23, bidlow=218.82, ask=223.8, askhigh=231.25, asklow=218.86, last=223.77, lasthigh=231.25, lastlow=218.82, volume_real=114.0, volumehigh_real=92998.0, volumelow_real=100.0, option_strike=0.0, point=0.01, trade_tick_value=0.01, trade_tick_value_profit=0.01, trade_tick_value_loss=0.01, trade_tick_size=0.01, trade_contract_size=1.0, trade_accrued_interest=0.0, trade_face_value=0.0, trade_liquidity_rate=1.0, volume_min=1.0, volume_max=1

In [7]:
res_mkt_ord = mtb.place_market_order(
    symbol="CHFSEK",
    vol=1.0,
    buy_sell="Buy",
    sl_pct=0.05,
    tp_pct=0.05
)

Order check FAILED for CHFSEK. retcode=10030 comment=Unsupported filling mode
Market order validation failed for CHFSEK


In [ ]:
df = mtb.get_position_df()
df2 = df.with_columns(pl.col("time").cast(pl.Datetime("ms")))
print(df2)